# Импорты и загрузка таблиц

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / "data" 
RATING_PATH = DATASET_PATH / "raw" / "ratings.csv"

ratings_df = pd.read_csv(RATING_PATH, encoding="latin-1")

ratings_df["timestamp"] = pd.to_datetime(ratings_df["timestamp"], unit="s")

In [3]:
MOVIES_PATH = DATASET_PATH / "raw" / "movies.csv"

movies_df = pd.read_csv(MOVIES_PATH, encoding="latin-1")

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
TRAIN_PATH = DATASET_PATH / "processed" / "train_20261020_161901.csv"
TEST_PATH = DATASET_PATH / "processed" / "test_20261020_161901.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# Подготовка индексов пользователей и фильмов

In [5]:
# Уникальные пользователи и фильмы, известные на обучении
user_ids = np.sort(train["userId"].unique())
movie_ids = np.sort(train["movieId"].unique())

# Перевод исходных ID в индексы от 0 до N-1
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

# Создаём копии, чтобы не изменять исходные таблицы
train_model = train.copy()
test_model = test.copy()

train_model["user_idx"] = train_model["userId"].map(user_to_idx)
train_model["movie_idx"] = train_model["movieId"].map(movie_to_idx)

test_model["user_idx"] = test_model["userId"].map(user_to_idx)
test_model["movie_idx"] = test_model["movieId"].map(movie_to_idx)

# Оставляем в тесте только пары, для которых известны оба эмбеддинга
test_known = test_model.dropna(subset=["user_idx", "movie_idx"]).copy()

train_model["user_idx"] = train_model["user_idx"].astype("int64")
train_model["movie_idx"] = train_model["movie_idx"].astype("int64")
test_known["user_idx"] = test_known["user_idx"].astype("int64")
test_known["movie_idx"] = test_known["movie_idx"].astype("int64")

# Разбиение обучающей части

In [6]:
fit_parts = []
val_parts = []

for _, group in train_model.groupby("user_idx", sort=False):
    group = group.sort_values("timestamp")
    split_idx = int(len(group) * 0.9)
    
    if len(group) >= 3:
        fit_parts.append(group.iloc[:split_idx])
        val_parts.append(group.iloc[split_idx:])
    else:
        fit_parts.append(group)
        
fit_df = pd.concat(fit_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)

# Подготовка жанровой матрицы

In [7]:
genres_by_movie = (movies_df.drop_duplicates("movieId").set_index("movieId")["genres"].to_dict())

genre_sets = {}

for movie_id in movie_ids:
    genres = genres_by_movie.get(movie_id, "")
    
    if pd.isna(genres) or genres == "(no genres listed":
        genre_sets[movie_id] = set()
    else:
        genre_sets[movie_id] = set(genres.split("|"))
        
genre_names = sorted({genre for genres in genre_sets.values() for genre in genres})

genre_to_idx = {genre: idx for idx, genre in enumerate(genre_names)}

genre_matrix = np.zeros((len(movie_ids), len(genre_names)), dtype=np.float32)

for movie_id, genres in genre_sets.items():
    movie_idx = movie_to_idx[movie_id]
    
    for genre in genres:
        genre_matrix[movie_idx, genre_to_idx[genre]] = 1.0
        
print("Фильмов:", genre_matrix.shape[0])
print("Жанров:", genre_matrix.shape[1])
print("Размер матрицы:", genre_matrix.shape)
print("Жанры:", genre_names)

Фильмов: 8230
Жанров: 20
Размер матрицы: (8230, 20)
Жанры: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


# Обучение жанровой bpr-модели

# Сравнение с бейзлайном и обычным bpr

# Финальная оценка выбранной модели на тесте